In [1]:
import warnings
warnings.simplefilter('ignore')

In [2]:
import os, sys
from typing import Optional

import polars as pl
import pandas as pd

REPO_DATASET_PATH = "/kaggle/input/olympiadlevelmaths4llm"

PACKAGES_PATH = "/kaggle/usr/lib/aimo3_packages_offline"

if not os.path.exists(REPO_DATASET_PATH):
    REPO_DATASET_PATH = "/kaggle/input/datasets/guyahonakpongbaguidi/olympiadlevelmaths4llm"
if not os.path.exists(PACKAGES_PATH):
    PACKAGES_PATH = "/kaggle/usr/lib/notebooks/guyahonakpongbaguidi/aimo3_packages_offline"

PROJECT_PATH = REPO_DATASET_PATH + "/OlympiadLevelMaths4LLM"
SRC_PATH = PROJECT_PATH + "/src"
BUILD_LIB_PATH = PROJECT_PATH + "/build/lib"

while BUILD_LIB_PATH in sys.path:
    sys.path.remove(BUILD_LIB_PATH)
if SRC_PATH in sys.path:
    sys.path.remove(SRC_PATH)
sys.path.insert(0, SRC_PATH)

# If this cell is re-run in a warm notebook, purge cached olympiad_llm modules so
# imports are forced to come from SRC_PATH instead of an installed/build copy.
for module_name in list(sys.modules):
    if module_name == "olympiad_llm" or module_name.startswith("olympiad_llm."):
        del sys.modules[module_name]

# --- Runtime / platform guards ---
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["TIKTOKEN_ENCODINGS_BASE"] = (
    PACKAGES_PATH + "/tiktoken_encodings"
)
os.environ["AIMO3_WHEELS_PATH"] = (
    PACKAGES_PATH + "/utils"
)
os.environ["AIMO3_PACKAGE_MANAGER"] = "pip"
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ["AIMO3_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE"] = "0"
os.environ["AIMO3_VLLM_ENABLE_PREFIX_CACHING"] = "0"

# =============================================================================
# V2 CONFIG — only env vars that AIMO3Config.from_env() actually reads
# Keep prompts compact so they do not overshadow short/easy problems.
# =============================================================================

# --- Prompts (override defaults from config.py) ---
os.environ["AIMO3_SYSTEM_PROMPT"] = """
Solve for the unique correct integer n with 0 <= n <= 99999.
Use the shortest exact path, not the longest explanation.
Identify the key structure first: invariant, modular constraint, factorization, recurrence, symmetry, exact counting formula, or an exact algebraic/coordinate reduction.
Small cases may suggest a conjecture, but they never justify the final answer.
Use Python only to test tiny cases, verify a derived formula, or perform the final exact computation.
Prefer exact integer or rational arithmetic throughout; avoid floats unless you can prove they cannot affect the final integer.
Avoid brute force unless the search space is genuinely tiny after pruning.
If one approach stalls, switch methods instead of extending a weak path.
Once n is established exactly, stop and return only \\boxed{n}.
""".strip()

os.environ["AIMO3_TOOL_PROMPT"] = """
Use this stateful Python notebook as an exact scratchpad.
Before coding, state in 1-2 lines what quantity you will compute and by what exact method.
Use exact integers, Fraction, modular arithmetic, factorizations, recurrences, combinatorial formulas, and symbolic simplification when helpful.
Treat tiny-case experiments and numerical checks as conjecture tests only, not proof.
Check complexity before loops; prefer closed forms, pruning, and exact formulas over large enumeration.
Keep code short, deterministic, and decisive. Use print() for every result you need to inspect.
Do not print FINAL_ANSWER until the integer is fully justified.
When the final integer is certain, print exactly FINAL_ANSWER=<n> on its own line and stop.
""".strip()

os.environ["AIMO3_PREFERENCE_PROMPT"] = """
Prefer an exact derivation over pattern matching.
For counting or divisibility, derive the exact formula, recurrence, or p-adic valuation, then verify it with Python.
For algebra and number theory, try modular constraints, factorization, and invariants before casework.
For geometry, move to an exact coordinate/vector/trig reduction only if it simplifies the problem.
Never guess from small cases.
""".strip()

os.environ["AIMO3_ANSWER_ONLY_PROMPT"] = """
Find the exact integer n with 0 <= n <= 99999.
Think silently. Do not explain. Do not guess from patterns.
Return only \\boxed{n}.
""".strip()


os.environ["AIMO3_REASONING_FRAMEWORK_ENABLED"] = "1"

# --- Model / server ---
# Set to "llama_cpp" for GGUF models; keep "vllm" for transformer checkpoints.
os.environ["AIMO3_INFERENCE_BACKEND"] = "vllm"
# Extra offline wheel names, comma-separated. Leave empty for the default stack.
# The installer also auto-adds llama-cpp-python when AIMO3_INFERENCE_BACKEND=llama_cpp.
os.environ["AIMO3_EXTRA_REQUIRED_LIBRARIES"] = ""
os.environ["AIMO3_MODEL_PATH"] = "/kaggle/input/gpt-oss-120b/transformers/default/1"
# os.environ["AIMO3_MODEL_PATH"] = "/kaggle/input/models/huikang/gpt-oss-120b-aimo3/transformers/160a/16"
os.environ["AIMO3_SERVED_MODEL_NAME"] = "gpt-oss"
os.environ["AIMO3_REUSE_EXISTING_SERVER"] = "1"
os.environ["AIMO3_SERVER_TIMEOUT"] = "180"
os.environ["AIMO3_REQUIRE_CUDA"] = "1"

# llama.cpp / GGUF knobs (used only when AIMO3_INFERENCE_BACKEND=llama_cpp)
os.environ["AIMO3_LLAMA_CPP_N_GPU_LAYERS"] = "-1"

# Cold-start speedup
os.environ["AIMO3_PRELOAD_MODEL_WEIGHTS"] = "1"
os.environ["AIMO3_PRELOAD_MODEL_WORKERS"] = "8"

# --- Display / tracing ---
os.environ["AIMO3_DISPLAY_CANDIDATES"] = "1"
os.environ["AIMO3_TRACE"] = "1"
os.environ["AIMO3_TRACE_ATTEMPTS"] = "1"
os.environ["AIMO3_TRACE_FULL_REASONING"] = "1"
os.environ["AIMO3_TRACE_ENV"] = "1"
os.environ["AIMO3_TRACE_ENV_PACKAGES"] = "sympy,numpy,mpmath,jupyter_client,ortools,z3-solver"
os.environ["AIMO3_TRACE_INCLUDE_PROBLEM_TEXT"] = "0"

# --- Core decoding / capacity ---
os.environ["AIMO3_SEED"] = "42"
os.environ["AIMO3_SEARCH_TOKENS"] = "128"
os.environ["AIMO3_MIN_TOKENS_BEFORE_STREAM_EXTRACTION"] = "1536"
os.environ["AIMO3_CONTEXT_TOKENS"] = "128000"
os.environ["AIMO3_BATCH_SIZE"] = "512"
os.environ["AIMO3_GPU_MEMORY_UTILIZATION"] = "0.9"
os.environ["AIMO3_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE"] = "0"

# --- Sandbox/tooling ---
os.environ["AIMO3_JUPYTER_TIMEOUT"] = "180"
os.environ["AIMO3_SANDBOX_TIMEOUT"] = "5"

# --- Time budgeting ---
os.environ["AIMO3_PROBLEMS_TOTAL"] = "50"
os.environ["AIMO3_NOTEBOOK_LIMIT"] = "17700"
os.environ["AIMO3_BASE_PROBLEM_TIMEOUT"] = "280"
os.environ["AIMO3_HIGH_PROBLEM_TIMEOUT"] = "1800"

# --- Attempt scheduling ---
os.environ["AIMO3_ATTEMPTS"] = "1"
os.environ["AIMO3_WORKERS"] = "2"
os.environ["AIMO3_TURNS"] = "64"
os.environ["AIMO3_EARLY_STOP"] = "3"
os.environ["AIMO3_EARLY_STOP_MIN_VERIFIED"] = "0"

# --- Extraction ---
os.environ["AIMO3_STRICT_FALLBACK_EXTRACTION"] = "1"

# --- Decoding knobs ---
os.environ["AIMO3_TEMPERATURE"] = "0.0"
os.environ["AIMO3_MIN_P"] = "0.02"
os.environ["AIMO3_TOP_P"] = "0.98"
os.environ["AIMO3_TOP_K"] = "-1"

# Verification stage removed in v2; keep notebook env focused on the live runtime.

# --- Time Management Approach ---
# - 'equal': use equal share of remaining time
# - 'base': use configured base_timeout_s for every problem
# - 'avg': use rolling average
# - 'cumulative': add carryover from previous problems
# - 'hybrid': take max(equal, avg, base) (default behavior)
os.environ["AIMO3_BUDGET_STRATEGY"] = "cumulative"
os.environ["AIMO3_BASE_TIMEOUT_S"] = ""  # unset to avoid overriding base_problem_timeout
os.environ["AIMO3_CARRYOVER_ENABLED"] = "1"
os.environ["AIMO3_CUMULATIVE_DISTRIBUTE"] = "0"

os.environ["AIMO3_WICKELGREN"] = "1"
os.environ["AIMO3_PORTFOLIO_ENABLED"] = "1"
os.environ["AIMO3_PORTFOLIO_SCOUT_ATTEMPTS"] = "2"
os.environ["AIMO3_PORTFOLIO_SUMMARY_MAX_CHARS"] = "2400"
os.environ["AIMO3_PORTFOLIO_TEMPERATURE_SCHEDULE"] = "0.0,0.05,0.18,0.30"
os.environ["AIMO3_TRACE_ATTEMPTS_MAX_CHARS"] = "60000"

os.environ["AIMO3_FILTER_TO_VERIFIED_IF_ANY"] = "0"
os.environ["AIMO3_ENTROPY_WEIGHTING"] = "0"
os.environ["AIMO3_RANKING_STRATEGY"] = "votes_then_verified"

# --- CPU retriever (v2, v1-compatible env names) ---
# Set this path to your mounted KB dir containing concepts.json or concepts.pkl
os.environ["AIMO3_RETRIEVER_ENABLED"] = "0"
os.environ["AIMO3_RETRIEVER_KB_PATH"] = "/kaggle/input/olympiadlevelmaths4llmdb/OlympiadLevelMaths4LLMDB/knowledge_base"
os.environ["AIMO3_RETRIEVER_CPU_ONLY"] = "1"
os.environ["AIMO3_RETRIEVER_TOP_K"] = "5"
os.environ["AIMO3_RETRIEVER_MIN_SCORE"] = "0.08"
os.environ["AIMO3_RETRIEVER_INCLUDE_EXAMPLES"] = "1"
os.environ["AIMO3_RETRIEVER_INCLUDE_DEFINITIONS"] = "1"
os.environ["AIMO3_RETRIEVER_WARMUP_ON_INIT"] = "1"
os.environ["AIMO3_RETRIEVER_MODEL_PATH"] = "/kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2"

# --- Compact agent memory (skill + failure hints) ---
os.environ["AIMO3_AGENT_MEMORY_ENABLED"] = "1"
os.environ["AIMO3_AGENT_MEMORY_PATH"] =  "" # REPO_DATASET_PATH + "/agent_memory.json"
os.environ["AIMO3_AGENT_MEMORY_SKILL_TOP_K"] = "2"
os.environ["AIMO3_AGENT_MEMORY_FAILURE_TOP_K"] = "2"
os.environ["AIMO3_AGENT_MEMORY_MIN_SCORE"] = "0.15"
os.environ["AIMO3_SEQUENTIAL_REPAIR_ENABLED"] = "1"
os.environ["AIMO3_SEQUENTIAL_REPAIR_MAX_ATTEMPTS"] = "1"
os.environ["AIMO3_SEQUENTIAL_REPAIR_MIN_ATTEMPTS"] = "1"
os.environ["AIMO3_SEQUENTIAL_REPAIR_ONLY_ON_TIMEOUT"] = "1"

os.environ["AIMO3_ADAPTIVE_BUDGET_FLEX_POOL_FRACTION"] = "0"

# --- safe repetition watchdog ---
os.environ["AIMO3_REPETITION_SIMILARITY_THRESHOLD"] = "0.93"
os.environ["AIMO3_REPETITION_SOFT_STREAK"] = "2"
os.environ["AIMO3_REPETITION_HARD_STREAK"] = "3"
os.environ["AIMO3_REPETITION_TOOL_REPEAT_HARD_STREAK"] = "2"
os.environ["AIMO3_REPETITION_MIN_CHARS"] = "100"
os.environ["AIMO3_SEQUENTIAL_REPAIR_ONLY_ON_TIMEOUT"] = "1"

# --- fast-exit methods ---
os.environ["AIMO3_EARLY_BOXED_EXIT_ENABLED"] = "0"
os.environ["AIMO3_TOOL_FINAL_ANSWER_MARKER_ENABLED"] = "1"


# --- Meta Learning ---
os.environ["AIMO3_META_LEARNING_ENABLED"] = "0"
os.environ["AIMO3_META_LEARNING_SIMILARITY_THRESHOLD"] = "0.3"

os.environ["AIMO3_Z3_TOOL_ENABLED"] = "0"

# Misc
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
os.environ["AIMO3_50_PROBLEMS_DATA_ENABLED"] = "0"


# --- Sandbow error silent ---
os.environ["AIMO3_SANDBOX_SILENCE_KERNEL_STDERR"] = "1"

# --- Final answer normalization / constraints. ---
os.environ["AIMO3_ANSWER_MODULUS"] = "100000"
os.environ["AIMO3_ANSWER_MIN"] = "0"
os.environ["AIMO3_ANSWER_MAX"] = "99999"

MODEL_PATH = os.getenv("AIMO3_MODEL_PATH", "")
print(f"Import path pinned to: {SRC_PATH}")
print(
    f"Config: BACKEND={os.environ.get('AIMO3_INFERENCE_BACKEND')}, EXTRA_REQUIRED_LIBRARIES={os.environ.get('AIMO3_EXTRA_REQUIRED_LIBRARIES')!r}"
)
print(
    f"Config: BASE_TIMEOUT={os.environ.get('AIMO3_BASE_PROBLEM_TIMEOUT')}s, HIGH_TIMEOUT={os.environ.get('AIMO3_HIGH_PROBLEM_TIMEOUT')}s"
)
print(
    f"Config: EARLY_STOP={os.environ.get('AIMO3_EARLY_STOP')}, MIN_VERIFIED={os.environ.get('AIMO3_EARLY_STOP_MIN_VERIFIED')}"
)
print(
    f"Config: ATTEMPTS={os.environ.get('AIMO3_ATTEMPTS')}, WORKERS={os.environ.get('AIMO3_WORKERS')}, TURNS={os.environ.get('AIMO3_TURNS')}"
)
print(
    f"Config: RANKING_STRATEGY={os.environ.get('AIMO3_RANKING_STRATEGY')}, FILTER_TO_VERIFIED_IF_ANY={os.environ.get('AIMO3_FILTER_TO_VERIFIED_IF_ANY')}"
)
print(
    f"Config: RETRIEVER_ENABLED={os.environ.get('AIMO3_RETRIEVER_ENABLED')}, RETRIEVER_KB_PATH={os.environ.get('AIMO3_RETRIEVER_KB_PATH')}"
)
print(
    "Config: CONTEXT_TOKENS="
    f"{os.environ.get('AIMO3_CONTEXT_TOKENS')}, BATCH_SIZE={os.environ.get('AIMO3_BATCH_SIZE')}, "
    f"GPU_MEM_UTIL={os.environ.get('AIMO3_GPU_MEMORY_UTILIZATION')}, "
    f"MAX_CUDAGRAPH={os.environ.get('AIMO3_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE')}, "
    f"PREFIX_CACHING={os.environ.get('AIMO3_VLLM_ENABLE_PREFIX_CACHING')}"
)
print(f"Config: PYTORCH_ALLOC_CONF={os.environ.get('PYTORCH_ALLOC_CONF')}")
MODEL_PATH

Import path pinned to: /kaggle/input/olympiadlevelmaths4llm/OlympiadLevelMaths4LLM/src
Config: BACKEND=vllm, EXTRA_REQUIRED_LIBRARIES=''
Config: BASE_TIMEOUT=280s, HIGH_TIMEOUT=1800s
Config: EARLY_STOP=3, MIN_VERIFIED=0
Config: ATTEMPTS=1, WORKERS=2, TURNS=64
Config: RANKING_STRATEGY=votes_then_verified, FILTER_TO_VERIFIED_IF_ANY=0
Config: RETRIEVER_ENABLED=0, RETRIEVER_KB_PATH=/kaggle/input/olympiadlevelmaths4llmdb/OlympiadLevelMaths4LLMDB/knowledge_base
Config: CONTEXT_TOKENS=128000, BATCH_SIZE=512, GPU_MEM_UTIL=0.9, MAX_CUDAGRAPH=0, PREFIX_CACHING=0
Config: PYTORCH_ALLOC_CONF=expandable_segments:True


'/kaggle/input/gpt-oss-120b/transformers/default/1'

In [3]:
# --- Nemotron v2 memory-fit overrides (single H100 80GB) ---
model_path_lower = os.environ.get("AIMO3_MODEL_PATH", "").lower()
is_nemotron = "nemotron" in model_path_lower

if is_nemotron:
    # Weight-only size can exceed single-GPU VRAM; use hybrid GPU+CPU offload.
    os.environ["AIMO3_VLLM_CPU_OFFLOAD_GB"] = "16"
    os.environ["AIMO3_VLLM_SWAP_SPACE_GB"] = "24"

    # Conservative serving defaults to improve startup success on 120B-class checkpoints.
    os.environ["AIMO3_CONTEXT_TOKENS"] = "8192"
    os.environ["AIMO3_BATCH_SIZE"] = "1"
    os.environ["AIMO3_GPU_MEMORY_UTILIZATION"] = "0.85"
    os.environ["AIMO3_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE"] = "0"
    os.environ["AIMO3_VLLM_ENABLE_CHUNKED_PREFILL"] = "0"
    # Workaround for known vLLM CPU-offload assertion path in v1 engine.
    os.environ["AIMO3_VLLM_USE_V1"] = "0"
else:
    # Explicit zero keeps behavior unchanged for non-Nemotron models.
    os.environ.setdefault("AIMO3_VLLM_CPU_OFFLOAD_GB", "0")
    os.environ.setdefault("AIMO3_VLLM_SWAP_SPACE_GB", "0")

print(
    "Config: NEMOTRON_OVERRIDES="
    f"{int(is_nemotron)}, CPU_OFFLOAD_GB={os.environ.get('AIMO3_VLLM_CPU_OFFLOAD_GB', '0')}, "
    f"SWAP_SPACE_GB={os.environ.get('AIMO3_VLLM_SWAP_SPACE_GB', '0')}, "
    f"CHUNKED_PREFILL={os.environ.get('AIMO3_VLLM_ENABLE_CHUNKED_PREFILL')}, "
    f"VLLM_USE_V1={os.environ.get('AIMO3_VLLM_USE_V1', '')}, "
    f"CONTEXT_TOKENS={os.environ.get('AIMO3_CONTEXT_TOKENS')}, "
    f"BATCH_SIZE={os.environ.get('AIMO3_BATCH_SIZE')}"
 )

Config: NEMOTRON_OVERRIDES=0, CPU_OFFLOAD_GB=0, SWAP_SPACE_GB=0, CHUNKED_PREFILL=None, VLLM_USE_V1=, CONTEXT_TOKENS=128000, BATCH_SIZE=512


In [4]:
# Final v2 runtime overrides for the actual submission run (portfolio + system2 profile).
# Goal: diversify attempts, exploit early useful work, and steer later attempts toward the most promising exact branches.
os.environ["AIMO3_ATTEMPTS"] = "4"
os.environ["AIMO3_WORKERS"] = "4"
os.environ["AIMO3_ANSWER_ONLY_ATTEMPTS"] = "0"
os.environ["AIMO3_EARLY_STOP"] = "3"
os.environ["AIMO3_EARLY_STOP_MIN_VERIFIED"] = "1"

# Keep ranking simple/robust first; add entropy back only if A/B shows gain.
os.environ["AIMO3_ENTROPY_WEIGHTING"] = "0"
os.environ["AIMO3_FILTER_TO_VERIFIED_IF_ANY"] = "0"
os.environ["AIMO3_RANKING_STRATEGY"] = "votes_then_verified"

# Sampling and serving fit (120B-class safe defaults).
os.environ["AIMO3_TEMPERATURE"] = "0.0"
os.environ["AIMO3_BATCH_SIZE"] = "32"
os.environ["AIMO3_VLLM_ENABLE_PREFIX_CACHING"] = "1"
os.environ["AIMO3_PORTFOLIO_ENABLED"] = "1"
os.environ["AIMO3_PORTFOLIO_SCOUT_ATTEMPTS"] = "2"
os.environ["AIMO3_PORTFOLIO_SUMMARY_MAX_CHARS"] = "2400"
os.environ["AIMO3_PORTFOLIO_TEMPERATURE_SCHEDULE"] = "0.0,0.05,0.18,0.30"
os.environ["AIMO3_SYSTEM2_ENABLED"] = "1"
os.environ["AIMO3_SYSTEM2_SCOUT_ATTEMPTS"] = "2"
os.environ["AIMO3_SYSTEM2_TOP_BRANCHES"] = "2"
os.environ["AIMO3_SYSTEM2_SUMMARY_MAX_CHARS"] = "2600"
os.environ["AIMO3_SYSTEM2_PRIOR_WEIGHT"] = "0.35"
os.environ["AIMO3_SYSTEM2_PROCESS_REWARD_WEIGHT"] = "1.00"
os.environ["AIMO3_SYSTEM2_DIVERSITY_WEIGHT"] = "0.30"
os.environ["AIMO3_SYSTEM2_ERROR_PENALTY_WEIGHT"] = "0.75"
os.environ["AIMO3_EARLY_BOXED_EXIT_ENABLED"] = "0"

# Timeouts: do NOT tie Jupyter timeout to per-problem timeout.
# Jupyter timeout is per tool execution; keep it bounded to avoid one call consuming the budget.
os.environ["AIMO3_JUPYTER_TIMEOUT"] = "20"
os.environ["AIMO3_SANDBOX_TIMEOUT"] = "4"

# Keep adaptive budget and repair available, but conservative.
os.environ["AIMO3_ADAPTIVE_BUDGET_FLEX_POOL_FRACTION"] = "0.10"
os.environ["AIMO3_SEQUENTIAL_REPAIR_ONLY_ON_TIMEOUT"] = "0"
os.environ["AIMO3_SEQUENTIAL_REPAIR_MAX_ATTEMPTS"] = "2"
os.environ["AIMO3_SEQUENTIAL_REPAIR_MIN_ATTEMPTS"] = "2"

# Enable compact memory hints from your bundled KB.
os.environ["AIMO3_AGENT_MEMORY_PATH"] = "" # REPO_DATASET_PATH + "/agent_memory.json"

print("Final runtime overrides applied (portfolio + system2 profile):")
print(
    f"  ATTEMPTS={os.environ.get('AIMO3_ATTEMPTS')}, WORKERS={os.environ.get('AIMO3_WORKERS')}, "
    f"ANSWER_ONLY_ATTEMPTS={os.environ.get('AIMO3_ANSWER_ONLY_ATTEMPTS')}"
)
print(
    f"  EARLY_STOP={os.environ.get('AIMO3_EARLY_STOP')}, "
    f"EARLY_STOP_MIN_VERIFIED={os.environ.get('AIMO3_EARLY_STOP_MIN_VERIFIED')}"
)
print(
    f"  RANKING_STRATEGY={os.environ.get('AIMO3_RANKING_STRATEGY')}, "
    f"FILTER_TO_VERIFIED_IF_ANY={os.environ.get('AIMO3_FILTER_TO_VERIFIED_IF_ANY')}, "
    f"ENTROPY_WEIGHTING={os.environ.get('AIMO3_ENTROPY_WEIGHTING')}"
)
print(
    f"  JUPYTER_TIMEOUT={os.environ.get('AIMO3_JUPYTER_TIMEOUT')}, "
    f"SANDBOX_TIMEOUT={os.environ.get('AIMO3_SANDBOX_TIMEOUT')}"
)
print(
    f"  PORTFOLIO_ENABLED={os.environ.get('AIMO3_PORTFOLIO_ENABLED')}, "
    f"SCOUT_ATTEMPTS={os.environ.get('AIMO3_PORTFOLIO_SCOUT_ATTEMPTS')}, "
    f"TEMP_SCHEDULE={os.environ.get('AIMO3_PORTFOLIO_TEMPERATURE_SCHEDULE')}"
)
print(
    f"  SYSTEM2_ENABLED={os.environ.get('AIMO3_SYSTEM2_ENABLED')}, "
    f"SCOUT_ATTEMPTS={os.environ.get('AIMO3_SYSTEM2_SCOUT_ATTEMPTS')}, "
    f"TOP_BRANCHES={os.environ.get('AIMO3_SYSTEM2_TOP_BRANCHES')}"
)
print(
    f"  SYSTEM2_WEIGHTS=(prior={os.environ.get('AIMO3_SYSTEM2_PRIOR_WEIGHT')}, "
    f"process={os.environ.get('AIMO3_SYSTEM2_PROCESS_REWARD_WEIGHT')}, "
    f"diversity={os.environ.get('AIMO3_SYSTEM2_DIVERSITY_WEIGHT')}, "
    f"error_penalty={os.environ.get('AIMO3_SYSTEM2_ERROR_PENALTY_WEIGHT')})"
)
print(
    f"  CONTEXT_TOKENS={os.environ.get('AIMO3_CONTEXT_TOKENS')}, "
    f"BATCH_SIZE={os.environ.get('AIMO3_BATCH_SIZE')}, "
    f"PREFIX_CACHING={os.environ.get('AIMO3_VLLM_ENABLE_PREFIX_CACHING')}"
)
print(f"  AGENT_MEMORY_PATH={os.environ.get('AIMO3_AGENT_MEMORY_PATH')}")

Final runtime overrides applied (stability-first):
  ATTEMPTS=4, WORKERS=4, ANSWER_ONLY_ATTEMPTS=0
  EARLY_STOP=3, EARLY_STOP_MIN_VERIFIED=1
  RANKING_STRATEGY=votes_then_verified, FILTER_TO_VERIFIED_IF_ANY=0, ENTROPY_WEIGHTING=0
  JUPYTER_TIMEOUT=15, SANDBOX_TIMEOUT=3
  CONTEXT_TOKENS=128000, BATCH_SIZE=32, PREFIX_CACHING=1
  AGENT_MEMORY_PATH=


In [5]:
# --- Optional full sandbox preload override ---
"""
Use this when you want COMPLETE control over what runs in each fresh sandbox kernel.
This overrides the default preload in src/olympiad_llm/aimo3/v2/sandbox.py.

How to use:
1) Set ENABLE_SANDBOX_PRELOAD_OVERRIDE = True
2) Edit SANDBOX_PRELOAD_OVERRIDE_CODE below
3) Keep this cell BEFORE solver initialization
"""

ENABLE_SANDBOX_PRELOAD_OVERRIDE = True

SANDBOX_PRELOAD_OVERRIDE_CODE = r'''
import sys
try:
    sys.set_int_max_str_digits(0)
except AttributeError:
    pass

import os
# _lean_bin = os.environ.get("AIMO3_LEAN_BIN_DIR")
# if _lean_bin and _lean_bin not in os.environ.get("PATH", ""):
#     os.environ["PATH"] = _lean_bin + os.pathsep + os.environ.get("PATH", "")

import math
import random
import itertools
import collections
import fractions
from fractions import Fraction

import numpy
import numpy as np
import sympy
import sympy as sp
import mpmath
import mpmath as mp

try:
    _mp_dps_raw = os.environ.get("AIMO3_SANDBOX_MPMATH_DPS", "32")
    _mp_dps = int(float(_mp_dps_raw))
    if _mp_dps <= 0:
        _mp_dps = 32
except Exception:
    _mp_dps = 32
mpmath.mp.dps = _mp_dps

try:
    import ortools  # noqa: F401
    from ortools.sat.python import cp_model  # noqa: F401
except Exception:
    pass

# try:
#     from z3 import *  # noqa: F401,F403
#     z3_available = True
# except Exception:
#     z3_available = False

# You can add ANY custom preload code below.
# Example:
# import statistics
# from math import comb
'''

if ENABLE_SANDBOX_PRELOAD_OVERRIDE:
    os.environ["AIMO3_SANDBOX_PRELOAD_OVERRIDE"] = SANDBOX_PRELOAD_OVERRIDE_CODE
    print("Config: AIMO3_SANDBOX_PRELOAD_OVERRIDE=enabled")
else:
    os.environ.pop("AIMO3_SANDBOX_PRELOAD_OVERRIDE", None)
    print("Config: AIMO3_SANDBOX_PRELOAD_OVERRIDE=disabled (using default preload)")

Config: AIMO3_SANDBOX_PRELOAD_OVERRIDE=enabled


In [6]:
from olympiad_llm.aimo3.v2.cleanup import environ_setup_parallel, wait_all

# Start environment setup (uninstall + offline install + model warmup) ALL IN PARALLEL.
# This overlaps pip install with model cache warmup, saving ~30-60s on cold start.
ENVIRON_SETUP = environ_setup_parallel(warm_model=True, model_workers=8)
print("Started parallel environment setup:")
print("  - pip uninstall conflicts (background)")
print("  - pip install required packages (background)")
print("  - Model weight cache warmup (background)")
print("Will wait for completion lazily when the solver is first needed.")

Started parallel environment setup:
  - pip uninstall conflicts (background)
  - pip install required packages (background)
  - Model weight cache warmup (background)
Will wait for completion lazily when the solver is first needed.


In [7]:
import inspect
import threading
from olympiad_llm.aimo3.v2.config import AIMO3Config
from olympiad_llm.aimo3.v2.runner import build_solver, run_kaggle_inference
from olympiad_llm.aimo3.v2.cleanup import wait_all

# IMPORTANT for Kaggle: don't block notebook execution on vLLM cold-start here.
# We'll build the solver lazily on the first predict() call.
config_module_path = inspect.getfile(AIMO3Config)
runner_module_path = inspect.getsourcefile(build_solver) or inspect.getfile(build_solver)
print(f"AIMO3Config imported from: {config_module_path}")
print(f"build_solver imported from: {runner_module_path}")
if "/src/olympiad_llm/" not in config_module_path:
    raise RuntimeError(
        "Notebook imported olympiad_llm from the wrong location. "
        "Re-run cell 2 first so SRC_PATH is first on sys.path and cached modules are cleared."
    )

cfg = AIMO3Config.from_env()
resolved_max_cudagraph = getattr(cfg, "vllm_max_cudagraph_capture_size", None)
resolved_prefix_caching = getattr(cfg, "vllm_enable_prefix_caching", None)
print(
    "Resolved cfg: "
    f"context_tokens={cfg.context_tokens}, batch_size={cfg.batch_size}, "
    f"gpu_memory_utilization={cfg.gpu_memory_utilization}, "
    f"max_cudagraph={resolved_max_cudagraph}, "
    f"prefix_caching={resolved_prefix_caching}, workers={cfg.workers}, attempts={cfg.attempts}"
)

expected_context_tokens = int(os.environ["AIMO3_CONTEXT_TOKENS"])
expected_batch_size = int(os.environ["AIMO3_BATCH_SIZE"])
expected_gpu_util = float(os.environ["AIMO3_GPU_MEMORY_UTILIZATION"])
expected_max_cudagraph = int(os.environ["AIMO3_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE"])
expected_prefix_caching = os.environ["AIMO3_VLLM_ENABLE_PREFIX_CACHING"].strip().lower() not in {"0", "false", "no"}

if cfg.context_tokens != expected_context_tokens or cfg.batch_size != expected_batch_size:
    raise RuntimeError(
        "Resolved AIMO3Config does not match notebook env overrides. "
        f"Expected context_tokens={expected_context_tokens}, batch_size={expected_batch_size}; "
        f"got context_tokens={cfg.context_tokens}, batch_size={cfg.batch_size}."
    )
if abs(cfg.gpu_memory_utilization - expected_gpu_util) > 1e-9:
    raise RuntimeError(
        "Resolved AIMO3Config does not match notebook env overrides for gpu_memory_utilization. "
        f"Expected {expected_gpu_util}, got {cfg.gpu_memory_utilization}."
    )
if resolved_max_cudagraph != expected_max_cudagraph:
    raise RuntimeError(
        "Resolved AIMO3Config does not match notebook env overrides for max cudagraph capture size. "
        f"Expected {expected_max_cudagraph}, got {resolved_max_cudagraph}."
    )
if resolved_prefix_caching != expected_prefix_caching:
    raise RuntimeError(
        "Resolved AIMO3Config does not match notebook env overrides for prefix caching. "
        f"Expected {expected_prefix_caching}, got {resolved_prefix_caching}."
    )

solver = None
_solver_lock = threading.Lock()

def get_solver():
    global solver
    if solver is not None:
        return solver
    with _solver_lock:
        if solver is not None:
            return solver
        # Wait for ALL parallel setup tasks to finish (pip + model warmup).
        if "ENVIRON_SETUP" in globals() and isinstance(ENVIRON_SETUP, dict):
            wait_all(ENVIRON_SETUP, timeout=180)
            print("✓ Parallel setup complete (pip + model cache warmup)")
        solver = build_solver(cfg)
        return solver

print("Lazy solver configured. Inference server can start now.")

AIMO3Config imported from: /kaggle/input/olympiadlevelmaths4llm/OlympiadLevelMaths4LLM/src/olympiad_llm/aimo3/v2/config.py
build_solver imported from: /kaggle/input/olympiadlevelmaths4llm/OlympiadLevelMaths4LLM/src/olympiad_llm/aimo3/v2/runner.py
Resolved cfg: context_tokens=128000, batch_size=32, gpu_memory_utilization=0.9, max_cudagraph=0, prefix_caching=True, workers=4, attempts=4
Lazy solver configured. Inference server can start now.


In [8]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    global correct_count, total_count, predictions, ground_truth
    
    question_id = id_.item(0)
    question_text = question.item(0)
    
    print("------")
    print(f"ID: {question_id}")
    
    # Build solver only when needed (keeps Kaggle inference server startup fast).
    s = get_solver()
    final_answer = s.solve_problem(question_text)
    predictions[question_id] = final_answer

    # Check accuracy if ground truth available (local runs only).
    total_count += 1
    if question_id in ground_truth:
        gt = ground_truth[question_id]
        is_correct = (final_answer == gt)
        if is_correct:
            correct_count += 1
        status = "✅" if is_correct else "❌"
        print(f"Answer: {final_answer} | Ground Truth: {gt} | {status}")
        print(f"📊 Running Accuracy: {correct_count}/{total_count} ({100*correct_count/total_count:.1f}%)")
    else:
        print(f"Answer: {final_answer}")
    
    print("------\n")
    
    return pl.DataFrame({'id': question_id, 'answer': final_answer})

In [9]:
from olympiad_llm.aimo3.prepare import prepare_reference_csv

In [10]:
# Load ground truth only for local testing (avoid delaying server start in competition reruns).
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    ground_truth = {}
elif int(os.environ.get('AIMO3_50_PROBLEMS_DATA_ENABLED')):
    df = pd.read_csv(REPO_DATASET_PATH + '/50problems.csv')
    df.insert(0, 'id', range(1, len(df) + 1))
    df.rename(columns={'Problem': 'problem', 'Answer': 'answer'}, inplace=True)
    print(df.head())
    df.to_csv('50problems.csv', index=False)
    ground_truth, _ = prepare_reference_csv(
        "50problems.csv",
    )
else:
    ground_truth, _ = prepare_reference_csv(
        "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv",
        # "/kaggle/input/olympiadlevelmaths4llmdb/inmo_1986.csv",
        # problem_ids=["dd7f5e", "86e8e5"],
        # problem_ids=["86e8e5"],
    )

# Track predictions for accuracy calculation
predictions = {}
correct_count = 0
total_count = 0

In [11]:
inference_server = run_kaggle_inference(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        ("reference.csv",)
        # ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )

------
ID: 9c1c5f
✓ Parallel setup complete (pip + model cache warmup)

Problem: Let $f \colon \mathbb{Z}_{\geq 1} \to \mathbb{Z}_{\geq 1}$ be a function such that for all positive integers $m$ and $n$, 
\begin{equation*}
    f(m) + f(n) = f(m + n + mn).
\end{equation*}
Across all functions $f$ such that $f(n) \leq 1000$ for all $n \leq 1000$, how many different values can $f(2024)$ take?

Budget: 318.60s | [Budget] 0/50 done | Remaining: 17700s | Flex: 1770s/1770s | Avg: 280s | Next: 319s | Extensions: 0



,Attempt,Answer,ToolVerified,PyCalls,Timeouts,PyErrors,Tokens,Entropy,Snippet,LastError
0,1,580,True,3,0,0,7713,None,"...3 and 5. Since we have a3 + a5 ≤ 1000, whic...",None
1,2,580,False,4,2,2,8021,None,"...on, then count values.\n\nWe also need to e...",[ERROR] Execution timed out after 15s. TIP: Fo...
2,4,580,True,2,0,0,7579,None,...). Could a or b be zero? The codomain is po...,None



Final Answer: 580 (votes=3, verified=2)

Answer: 580 | Ground Truth: 580 | ✅
📊 Running Accuracy: 1/1 (100.0%)
------

------
ID: dd7f5e

Problem: Let $\mathcal{F}$ be the set of functions $\alpha \colon \mathbb{Z}\to \mathbb{Z}$ for which there are only finitely many $n \in \mathbb{Z}$ such that $\alpha(n) \neq 0$. 

For two functions $\alpha$ and $\beta$ in $\mathcal{F}$, define their product $\alpha\star\beta$ to be $\sum\limits_{n\in\mathbb{Z}} \alpha(n)\cdot \beta(n)$. Also, for $n\in\mathbb{Z}$, define a shift operator $S_n \colon \mathcal{F}\to \mathcal{F}$ by $S_n(\alpha)(t)=\alpha(t+n)$ for all $t \in \mathbb{Z}$.

A function $\alpha \in \mathcal{F}$ is called \emph{shifty} if 
\begin{itemize}
    \item $\alpha(m)=0$ for all integers $m<0$ and $m>8$ and
    \item There exists $\beta \in \mathcal{F}$ and integers $k \neq l$ such that for all $n \in \mathbb{Z}$
    \begin{equation*}
        S_n(\alpha)\star\beta =
        \begin{cases}
            1 & n \in \{k,l\} \\
          

,Attempt,Answer,ToolVerified,PyCalls,Timeouts,PyErrors,Tokens,Entropy,Snippet,LastError
0,1,160,True,9,0,0,14893,None,...s primitive. So fine.\n\nNow we need to pro...,None
1,2,160,False,14,3,4,19450,None,"... The sum must be ≤ 8. For each such f, ther...",----------------------------------------------...
2,3,266,True,7,0,0,14569,None,"...ger from 0 to 8 - deg(D), giving 9 - deg(D)...",None
3,4,160,False,23,0,1,20540,None,"...omial x^k + x^l, leading to α(x) = x^e D(x)...",----------------------------------------------...



Final Answer: 160 (votes=3, verified=1)

Answer: 160 | Ground Truth: 160 | ✅
📊 Running Accuracy: 2/2 (100.0%)
------

------
ID: 86e8e5

Problem: Let $n \geq 6$ be a positive integer. We call a positive integer $n$-Norwegian if it has three distinct positive divisors whose sum is equal to $n$. Let $f(n)$ denote the smallest $n$-Norwegian positive integer. Let $M=3^{2025!}$ and for a non-negative integer $c$ define 
\begin{equation*}
    g(c)=\frac{1}{2025!}\left\lfloor \frac{2025! f(M+c)}{M}\right\rfloor.
\end{equation*}
We can write 
\begin{equation*}
    g(0)+g(4M)+g(1848374)+g(10162574)+g(265710644)+g(44636594)=\frac{p}{q}
\end{equation*}
where $p$ and $q$ are coprime positive integers. What is the remainder when $p+q$ is divided by $99991$?

Budget: 560.33s | [Budget] 2/50 done | Remaining: 17305s | Flex: 1770s/1770s | Avg: 198s | Next: 324s | Extensions: 0

[Adaptive] Running sequential repair pass (1 max attempts)...


...


Final Answer: 32193 (votes=4, verified=1)

Answer: 32193 | Ground Truth: 32193 | ✅
📊 Running Accuracy: 7/8 (87.5%)
------

------
ID: 0e644e

Problem: Let $ABC$ be an acute-angled triangle with integer side lengths and $AB<AC$. Points $D$ and $E$ lie on segments $BC$ and $AC$, respectively, such that $AD=AE=AB$. Line $DE$ intersects $AB$ at $X$. Circles $BXD$ and $CED$ intersect for the second time at $Y \neq D$. Suppose that $Y$ lies on line $AD$. There is a unique such triangle with minimal perimeter. This triangle has side lengths $a=BC$, $b=CA$, and $c=AB$. Find the remainder when $abc$ is divided by $10^{5}$.

Budget: 411.58s | [Budget] 8/50 done | Remaining: 15243s | Flex: 1770s/1770s | Avg: 307s | Next: 321s | Extensions: 0



,Attempt,Answer,ToolVerified,PyCalls,Timeouts,PyErrors,Tokens,Entropy,Snippet,LastError
0,2,336,True,15,0,0,20101,None,...00 = 336.\n\nBut we need to be careful: The...,None
1,3,336,True,17,0,0,28198,None,...b + c). Then find integer solutions with b ...,None
2,4,336,True,13,0,0,20426,None,...Actually we found only one triple with peri...,None



Final Answer: 336 (votes=3, verified=3)

Answer: 336 | Ground Truth: 336 | ✅
📊 Running Accuracy: 8/9 (88.9%)
------

------
ID: a295e9

Problem: A $500 \times 500$ square is divided into $k$ rectangles, each having integer side lengths. Given that no two of these rectangles have the same perimeter, the largest possible value of $k$ is $\mathcal{K}$. What is the remainder when $k$ is divided by $10^{5}$?

Budget: 508.43s | [Budget] 9/50 done | Remaining: 15021s | Flex: 1770s/1770s | Avg: 298s | Next: 323s | Extensions: 0

[Adaptive] Running sequential repair pass (1 max attempts)...


,Attempt,Answer,ToolVerified,PyCalls,Timeouts,PyErrors,Tokens,Entropy,Snippet,LastError
0,2,155.0,True,3,0,0,20142,None,"... divided by 10^5. Since k = 155 < 100000, r...",None
1,4,156.0,False,18,4,4,24676,None,...plies k <= 156. So the theoretical bound is...,[ERROR] Execution timed out after 15s. TIP: Fo...
2,1,NaN,False,12,2,2,51268,None,...m_i w_i * n_i for given m by setting n_big ...,[ERROR] Execution timed out after 120s.
3,3,NaN,False,22,5,6,24845,None,"...\n return A, B\n else:\n r...",[ERROR] Execution timed out after 120s.



Final Answer: 155 (votes=1, verified=1)

Answer: 155 | Ground Truth: 520 | ❌
📊 Running Accuracy: 8/10 (80.0%)
------

